In [6]:
"""
BAT Annotation Agreement Analysis
==================================
Compares three annotation files (LLM, Nadia, Jihyun) on the four BAT
constructs (EX, EMO, COG, MD).

Steps:
1. Sanity check: confirm all three files cover the same set of posts
   (matched on post_id + comment_id, NOT on row order, since files can
   be sorted differently).
2. Normalize value formats (some files use "0"/"1" instead of "NO"/"YES").
3. For each construct (EX, EMO, COG, MD):
   - YES/NO counts per annotator
   - Pairwise agreement (LLM-vs-Nadia, LLM-vs-Jihyun, Nadia-vs-Jihyun)
   - Three-way (all agree / not all agree) agreement
   - Cohen's kappa for each pair
   - Fleiss' kappa across all three annotators
   - List of disagreement rows for inspection
4. Overall bat_score agreement (exact match across all three).
5. Writes a text report + a CSV of per-row/per-construct disagreements.

Usage:
    python3 bat_agreement_analysis.py
Edit the FILES dict below if filenames/paths change.
"""

import pandas as pd
import numpy as np
from itertools import combinations

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
FILES = {
    # "LLM": "bat_post20_LLM.csv",
    "LLM": "bat_sample_60_llm.xlsx",
    "Nadia": "bat_sample_60_Nadia.xlsx",
    "Jihyun": "bat_sample_60_jiHyun.xlsx",
}
CONSTRUCTS = ["EX", "EMO", "COG", "MD"]
OUTPUT_REPORT = "bat_agreement_report_sample60.txt"
OUTPUT_DISAGREEMENTS_CSV = "bat_disagreements_sample60.csv"

ANNOTATORS = list(FILES.keys())
PAIRS = list(combinations(ANNOTATORS, 2))
REQUIRED_COLS = ["post_id", "comment_id"] + CONSTRUCTS


# ---------------------------------------------------------------------
# LOAD + NORMALIZE
# ---------------------------------------------------------------------
def load_any(path):
    """Load a CSV or XLSX into a DataFrame.

    For XLSX files, scans the first several rows to find the real header
    row (the row containing 'post_id' and the construct columns) instead
    of blindly trusting row 0. Some exported files have a stray row above
    the real header (e.g. a leftover sample-count label like the '60'
    that showed up in bat_sample_60_jiHyun.xlsx), which pushes pandas'
    default header detection off by one and produces 'Unnamed: N' columns.
    """
    if not path.lower().endswith((".xlsx", ".xls")):
        return pd.read_csv(path)

    raw = pd.read_excel(path, header=None, nrows=10)

    header_row = 0
    target = {"post_id"} | {c.lower() for c in CONSTRUCTS}
    for i in range(len(raw)):
        row_values = {str(v).strip().lower() for v in raw.iloc[i].tolist() if pd.notna(v)}
        if target.issubset(row_values) or "post_id" in row_values:
            header_row = i
            break

    if header_row != 0:
        print(f"    NOTE: header row auto-detected at Excel row {header_row + 1} "
              f"(row 1 looked like a stray/label row, not the real header)")

    return pd.read_excel(path, header=header_row)


def load_and_normalize(name, path):
    df = load_any(path)

    # Normalize header whitespace/case so "Post_ID", " post_id ", "POST_ID"
    # etc. all resolve to the expected column name.
    rename_map = {}
    lower_stripped = {c: str(c).strip().lower() for c in df.columns}
    for wanted in REQUIRED_COLS:
        for orig, cleaned in lower_stripped.items():
            if cleaned == wanted.lower() and orig != wanted:
                rename_map[orig] = wanted
    if rename_map:
        print(f"  NOTE: [{name}] auto-matched column header(s): {rename_map}")
        df = df.rename(columns=rename_map)

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"\n\n[{name}] file '{path}' is missing required column(s): {missing}\n"
            f"  Columns actually found in this file: {df.columns.tolist()}\n"
            f"  Fix the header in the source file (or extend the rename_map "
            f"logic above) and re-run.\n"
        )

    df["comment_id"] = df["comment_id"].fillna("NONE")
    df["key"] = df["post_id"].astype(str) + "_" + df["comment_id"].astype(str)

    normalize_map = {"YES": "YES", "NO": "NO", "1": "YES", "0": "NO",
                      "1.0": "YES", "0.0": "NO"}
    zero_one_flags = 0
    for c in CONSTRUCTS:
        raw = df[c].astype(str).str.strip().str.upper()
        zero_one_flags += raw.isin(["0", "1", "0.0", "1.0"]).sum()
        df[c] = raw.map(normalize_map)
        unmapped = df[c].isna()
        if unmapped.any():
            print(f"  WARNING [{name}] unmapped values in {c}: "
                  f"{raw[unmapped].unique().tolist()}")

    if zero_one_flags:
        print(f"  NOTE: [{name}] had {zero_one_flags} cell(s) coded as "
              f"0/1 instead of NO/YES — normalized automatically.")

    return df.set_index("key")


def cohens_kappa(a, b):
    """Simple 2-rater, 2-category (YES/NO) Cohen's kappa."""
    n = len(a)
    po = (a == b).mean()
    pa_yes = (a == "YES").mean()
    pb_yes = (b == "YES").mean()
    pe = pa_yes * pb_yes + (1 - pa_yes) * (1 - pb_yes)
    if pe == 1:
        return 1.0
    return (po - pe) / (1 - pe)


def fleiss_kappa(rows):
    """
    rows: list of lists, each inner list has the categorical rating from
    each of the N raters for one item (categories: YES/NO).
    """
    categories = ["YES", "NO"]
    n_items = len(rows)
    n_raters = len(rows[0])

    # n_ij matrix: items x categories
    mat = np.zeros((n_items, len(categories)))
    for i, row in enumerate(rows):
        for val in row:
            j = categories.index(val)
            mat[i, j] += 1

    p_j = mat.sum(axis=0) / (n_items * n_raters)
    P_i = ((mat ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()
    P_e = (p_j ** 2).sum()
    if P_e == 1:
        return 1.0
    return (P_bar - P_e) / (1 - P_e)


def kappa_label(k):
    if k < 0:
        return "poor (worse than chance)"
    elif k < 0.20:
        return "slight"
    elif k < 0.40:
        return "fair"
    elif k < 0.60:
        return "moderate"
    elif k < 0.80:
        return "substantial"
    else:
        return "almost perfect"


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
def main():
    lines = []  # report text buffer
    def log(msg=""):
        print(msg)
        lines.append(msg)

    log("=" * 70)
    log("BAT ANNOTATION AGREEMENT ANALYSIS")
    log("=" * 70)

    # ---- Load ----
    log("\n--- Loading files & normalizing values ---")
    dfs = {}
    for name, path in FILES.items():
        print(f"Loading {name} ({path})...")
        dfs[name] = load_and_normalize(name, path)

    # ---- Sanity check: same rows across files ----
    log("\n" + "=" * 70)
    log("STEP 1: SANITY CHECK — do all files cover the same posts?")
    log("=" * 70)

    keysets = {name: set(df.index) for name, df in dfs.items()}
    all_match = True
    for a, b in PAIRS:
        same = keysets[a] == keysets[b]
        all_match &= same
        log(f"  {a} vs {b}: {'MATCH' if same else 'MISMATCH'}")
        if not same:
            only_a = keysets[a] - keysets[b]
            only_b = keysets[b] - keysets[a]
            if only_a:
                log(f"    Rows only in {a}: {sorted(only_a)}")
            if only_b:
                log(f"    Rows only in {b}: {sorted(only_b)}")

    for name, df in dfs.items():
        log(f"  Row count [{name}]: {len(df)} (unique keys: {df.index.nunique()})")

    if all_match:
        log("\n  RESULT: All three files annotate the exact same set of posts. Good.")
    else:
        log("\n  RESULT: MISMATCH DETECTED. Only overlapping posts will be used "
            "for agreement analysis below.")

    common_keys = sorted(set.intersection(*keysets.values()))
    log(f"\n  Using {len(common_keys)} common posts for all analyses below.")

    # Check row order (informational only, since we merge on key not position)
    orders_equal = all(
        list(dfs[a].index) == list(dfs[b].index) for a, b in PAIRS
        if set(dfs[a].index) == set(dfs[b].index)
    )
    log(f"  Row order identical across files: {orders_equal} "
        f"(not an issue — analysis merges on post_id/comment_id, not row position).")

    # Align all dataframes to common_keys, same order
    aligned = {name: df.loc[common_keys] for name, df in dfs.items()}

    # ---- Per-construct analysis ----
    log("\n" + "=" * 70)
    log("STEP 2: PER-CONSTRUCT YES/NO COUNTS & AGREEMENT")
    log("=" * 70)

    disagreement_rows = []

    for construct in CONSTRUCTS:
        log(f"\n--- {construct} ---")

        # Counts per annotator
        log("  YES / NO counts:")
        for name in ANNOTATORS:
            vc = aligned[name][construct].value_counts()
            yes = vc.get("YES", 0)
            no = vc.get("NO", 0)
            log(f"    {name:8s}: YES={yes:3d}  NO={no:3d}")

        # Pairwise agreement
        log("  Pairwise agreement:")
        for a, b in PAIRS:
            sa = aligned[a][construct]
            sb = aligned[b][construct]
            agree_mask = sa == sb
            pct = agree_mask.mean() * 100
            kappa = cohens_kappa(sa, sb)
            log(f"    {a:8s} vs {b:8s}: {agree_mask.sum():3d}/{len(sa)} "
                f"agree ({pct:5.1f}%)   Cohen's kappa = {kappa:.3f} "
                f"({kappa_label(kappa)})")

        # Three-way agreement
        stacked = pd.concat(
            [aligned[name][construct].rename(name) for name in ANNOTATORS],
            axis=1
        )
        all_agree_mask = stacked.nunique(axis=1) == 1
        n_all_agree = all_agree_mask.sum()
        n_total = len(stacked)
        log(f"  Three-way agreement (all {len(ANNOTATORS)} annotators match): "
            f"{n_all_agree}/{n_total} ({n_all_agree/n_total*100:.1f}%)")

        # Fleiss' kappa
        rows = stacked.values.tolist()
        fk = fleiss_kappa(rows)
        log(f"  Fleiss' kappa (all {len(ANNOTATORS)} annotators): "
            f"{fk:.3f} ({kappa_label(fk)})")

        # Record disagreement rows for CSV export
        for key in stacked.index[~all_agree_mask]:
            row = {"post_id_key": key, "construct": construct}
            for name in ANNOTATORS:
                row[name] = stacked.loc[key, name]
            disagreement_rows.append(row)

    # ---- bat_score agreement ----
    log("\n" + "=" * 70)
    log("STEP 3: OVERALL bat_score AGREEMENT")
    log("=" * 70)
    score_stacked = pd.concat(
        [aligned[name]["bat_score"].rename(name) for name in ANNOTATORS],
        axis=1
    )
    exact_match = score_stacked.nunique(axis=1) == 1
    log(f"  Exact bat_score match across all annotators: "
        f"{exact_match.sum()}/{len(score_stacked)} "
        f"({exact_match.mean()*100:.1f}%)")
    for a, b in PAIRS:
        diff = (score_stacked[a] - score_stacked[b]).abs()
        log(f"  {a:8s} vs {b:8s}: mean abs diff = {diff.mean():.2f}, "
            f"exact match = {(diff == 0).mean()*100:.1f}%")

    # ---- Summary table ----
    log("\n" + "=" * 70)
    log("STEP 4: SUMMARY TABLE (three-way agreement % by construct)")
    log("=" * 70)
    summary = {}
    for construct in CONSTRUCTS:
        stacked = pd.concat(
            [aligned[name][construct].rename(name) for name in ANNOTATORS],
            axis=1
        )
        pct = (stacked.nunique(axis=1) == 1).mean() * 100
        summary[construct] = pct
    for construct, pct in summary.items():
        log(f"  {construct:5s}: {pct:5.1f}% three-way agreement")

    # ---- Write outputs ----
    with open(OUTPUT_REPORT, "w") as f:
        f.write("\n".join(lines))

    if disagreement_rows:
        pd.DataFrame(disagreement_rows).to_csv(OUTPUT_DISAGREEMENTS_CSV, index=False)
        log(f"\nDisagreement detail written to: {OUTPUT_DISAGREEMENTS_CSV} "
            f"({len(disagreement_rows)} disagreement rows)")
    else:
        log("\nNo disagreements found — no disagreement CSV written.")

    log(f"Full report written to: {OUTPUT_REPORT}")


if __name__ == "__main__":
    main()

BAT ANNOTATION AGREEMENT ANALYSIS

--- Loading files & normalizing values ---
Loading LLM (bat_sample_60_llm.xlsx)...
Loading Nadia (bat_sample_60_Nadia.xlsx)...
  NOTE: [Nadia] had 240 cell(s) coded as 0/1 instead of NO/YES — normalized automatically.
Loading Jihyun (bat_sample_60_jiHyun.xlsx)...
    NOTE: header row auto-detected at Excel row 2 (row 1 looked like a stray/label row, not the real header)
  NOTE: [Jihyun] had 240 cell(s) coded as 0/1 instead of NO/YES — normalized automatically.

STEP 1: SANITY CHECK — do all files cover the same posts?
  LLM vs Nadia: MATCH
  LLM vs Jihyun: MATCH
  Nadia vs Jihyun: MATCH
  Row count [LLM]: 60 (unique keys: 59)
  Row count [Nadia]: 60 (unique keys: 59)
  Row count [Jihyun]: 60 (unique keys: 59)

  RESULT: All three files annotate the exact same set of posts. Good.

  Using 59 common posts for all analyses below.
  Row order identical across files: True (not an issue — analysis merges on post_id/comment_id, not row position).

STEP 2: PE

In [5]:
"""
Quick diagnostic: run this on the file that's failing (bat_sample_60_jiHyun.xlsx)
to see what pandas actually loaded, before running the full agreement script.
"""
import pandas as pd

PATH = "bat_sample_60_jiHyun.xlsx"

# 1. What sheets exist?
xl = pd.ExcelFile(PATH)
print("Sheets found:", xl.sheet_names)

# 2. What does the default (first) sheet look like?
df = pd.read_excel(PATH)
print("\nColumns (default sheet):")
for c in df.columns:
    print(f"  {repr(c)}")

print("\nFirst 3 rows:")
print(df.head(3))

Sheets found: ['bat_sample_60']

Columns (default sheet):
  'Unnamed: 0'
  'Unnamed: 1'
  'Unnamed: 2'
  60
  'Unnamed: 4'
  'Unnamed: 5'
  'Unnamed: 6'
  'Unnamed: 7'
  'Unnamed: 8'
  'Unnamed: 9'
  'Unnamed: 10'
  'Unnamed: 11'
  'Unnamed: 12'
  'Unnamed: 13'
  'Unnamed: 14'
  'Unnamed: 15'
  'Unnamed: 16'

First 3 rows:
  Unnamed: 0 Unnamed: 1  Unnamed: 2  \
0   row_type    post_id  comment_id   
1       post    1r13xch         NaN   
2       post    1dfaa19         NaN   

                                                  60 Unnamed: 4 Unnamed: 5  \
0                                               text         EX        EMO   
1  Nuke Hyper-V cluster and start over?\nHello al...          1          0   
2  Brain Fog and Dopamine Seeking brhaviour impac...          1          0   

  Unnamed: 6 Unnamed: 7 Unnamed: 8    Unnamed: 9    Unnamed: 10  \
0        COG         MD  bat_score  EX_reasoning  EMO_reasoning   
1          0          0          1           NaN            NaN   
2   

In [2]:
"""
Two-Way Agreement Analysis: Nadia vs. Jihyun
==============================================
Standalone script for human-human inter-rater agreement on the BAT
constructs (EX, EMO, COG, MD), separate from the three-way LLM comparison.

Reports:
  1. Sanity check: same posts in both files (matched on post_id + comment_id).
  2. Per-construct: YES/NO counts, % agreement, Cohen's kappa.
  3. Per-post exact match: how often Nadia and Jihyun agree on ALL 4
     constructs for the same post (all-or-nothing).
  4. Pooled overall: all (post x construct) decisions treated as one long
     vector of YES/NO calls -> one overall % agreement + one overall
     Cohen's kappa across all four constructs combined.
  5. bat_score agreement (exact match + mean absolute difference).
  6. List of disagreement rows for inspection.

Usage:
    python3 twoway_agreement_nadia_jihyun.py
Edit the FILES dict below if filenames/paths change.
"""

import pandas as pd

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
FILES = {
    "Nadia": "merged_post40_Nadia.csv",
    "Jihyun": "merged_post40_jiHyun.csv",
}
CONSTRUCTS = ["EX", "EMO", "COG", "MD"]
OUTPUT_REPORT = "twoway_agreement_report.txt"
OUTPUT_DISAGREEMENTS_CSV = "twoway_disagreements.csv"

RATER_A, RATER_B = list(FILES.keys())  # e.g. "Nadia", "Jihyun"


# ---------------------------------------------------------------------
# LOAD + NORMALIZE
# ---------------------------------------------------------------------
def load_and_normalize(name, path):
    df = pd.read_csv(path)
    df["comment_id"] = df["comment_id"].fillna("NONE")
    df["key"] = df["post_id"].astype(str) + "_" + df["comment_id"].astype(str)

    normalize_map = {"YES": "YES", "NO": "NO", "1": "YES", "0": "NO",
                      "1.0": "YES", "0.0": "NO"}
    zero_one_flags = 0
    for c in CONSTRUCTS:
        raw = df[c].astype(str).str.strip().str.upper()
        zero_one_flags += raw.isin(["0", "1", "0.0", "1.0"]).sum()
        df[c] = raw.map(normalize_map)
        unmapped = df[c].isna()
        if unmapped.any():
            print(f"  WARNING [{name}] unmapped values in {c}: "
                  f"{raw[unmapped].unique().tolist()}")

    if zero_one_flags:
        print(f"  NOTE: [{name}] had {zero_one_flags} cell(s) coded as "
              f"0/1 instead of NO/YES — normalized automatically.")

    return df.set_index("key")


def cohens_kappa(a, b):
    """2-rater, 2-category (YES/NO) Cohen's kappa."""
    po = (a == b).mean()
    pa_yes = (a == "YES").mean()
    pb_yes = (b == "YES").mean()
    pe = pa_yes * pb_yes + (1 - pa_yes) * (1 - pb_yes)
    if pe == 1:
        return 1.0
    return (po - pe) / (1 - pe)


def kappa_label(k):
    if k < 0:
        return "poor (worse than chance)"
    elif k < 0.20:
        return "slight"
    elif k < 0.40:
        return "fair"
    elif k < 0.60:
        return "moderate"
    elif k < 0.80:
        return "substantial"
    else:
        return "almost perfect"


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
def main():
    lines = []
    def log(msg=""):
        print(msg)
        lines.append(msg)

    log("=" * 70)
    log(f"TWO-WAY AGREEMENT ANALYSIS: {RATER_A} vs. {RATER_B}")
    log("=" * 70)

    # ---- Load ----
    log("\n--- Loading files & normalizing values ---")
    dfs = {}
    for name, path in FILES.items():
        print(f"Loading {name} ({path})...")
        dfs[name] = load_and_normalize(name, path)

    # ---- Sanity check ----
    log("\n" + "=" * 70)
    log("STEP 1: SANITY CHECK — do both files cover the same posts?")
    log("=" * 70)

    keys_a = set(dfs[RATER_A].index)
    keys_b = set(dfs[RATER_B].index)
    same = keys_a == keys_b
    log(f"  {RATER_A} vs {RATER_B}: {'MATCH' if same else 'MISMATCH'}")
    if not same:
        only_a = keys_a - keys_b
        only_b = keys_b - keys_a
        if only_a:
            log(f"    Rows only in {RATER_A}: {sorted(only_a)}")
        if only_b:
            log(f"    Rows only in {RATER_B}: {sorted(only_b)}")

    log(f"  Row count [{RATER_A}]: {len(dfs[RATER_A])}")
    log(f"  Row count [{RATER_B}]: {len(dfs[RATER_B])}")

    common_keys = sorted(keys_a & keys_b)
    log(f"\n  Using {len(common_keys)} common posts for the analysis below.")

    a_df = dfs[RATER_A].loc[common_keys]
    b_df = dfs[RATER_B].loc[common_keys]

    # ---- Per-construct agreement ----
    log("\n" + "=" * 70)
    log("STEP 2: PER-CONSTRUCT YES/NO COUNTS & AGREEMENT")
    log("=" * 70)

    disagreement_rows = []
    per_construct_kappa = {}

    for construct in CONSTRUCTS:
        log(f"\n--- {construct} ---")

        vc_a = a_df[construct].value_counts()
        vc_b = b_df[construct].value_counts()
        log(f"  {RATER_A:8s}: YES={vc_a.get('YES', 0):3d}  NO={vc_a.get('NO', 0):3d}")
        log(f"  {RATER_B:8s}: YES={vc_b.get('YES', 0):3d}  NO={vc_b.get('NO', 0):3d}")

        sa, sb = a_df[construct], b_df[construct]
        agree_mask = sa == sb
        pct = agree_mask.mean() * 100
        kappa = cohens_kappa(sa, sb)
        per_construct_kappa[construct] = kappa

        log(f"  Agreement: {agree_mask.sum()}/{len(sa)} ({pct:.1f}%)   "
            f"Cohen's kappa = {kappa:.3f} ({kappa_label(kappa)})")

        for key in sa.index[~agree_mask]:
            disagreement_rows.append({
                "post_id_key": key,
                "construct": construct,
                RATER_A: sa.loc[key],
                RATER_B: sb.loc[key],
            })

    # ---- Per-post exact match (all 4 constructs) ----
    log("\n" + "=" * 70)
    log("STEP 3: PER-POST EXACT MATCH (all 4 constructs agree, all-or-nothing)")
    log("=" * 70)
    stacked = pd.concat(
        [a_df[CONSTRUCTS].add_suffix(f"_{RATER_A}"),
         b_df[CONSTRUCTS].add_suffix(f"_{RATER_B}")],
        axis=1
    )
    exact_post_match = pd.Series(
        [all(a_df.loc[k, c] == b_df.loc[k, c] for c in CONSTRUCTS) for k in common_keys],
        index=common_keys
    )
    n_exact = exact_post_match.sum()
    log(f"  Posts where {RATER_A} and {RATER_B} agree on ALL 4 constructs: "
        f"{n_exact}/{len(common_keys)} ({n_exact/len(common_keys)*100:.1f}%)")

    # ---- Pooled overall (all post x construct decisions as one vector) ----
    log("\n" + "=" * 70)
    log("STEP 4: POOLED OVERALL AGREEMENT (all constructs combined)")
    log("=" * 70)
    a_pooled = pd.concat([a_df[c] for c in CONSTRUCTS], ignore_index=True)
    b_pooled = pd.concat([b_df[c] for c in CONSTRUCTS], ignore_index=True)
    n_pooled = len(a_pooled)
    pooled_agree = (a_pooled == b_pooled).sum()
    pooled_pct = pooled_agree / n_pooled * 100
    pooled_kappa = cohens_kappa(a_pooled, b_pooled)

    log(f"  Pooled decisions: {n_pooled} ({len(common_keys)} posts x {len(CONSTRUCTS)} constructs)")
    log(f"  Overall agreement: {pooled_agree}/{n_pooled} ({pooled_pct:.1f}%)")
    log(f"  Overall Cohen's kappa (pooled): {pooled_kappa:.3f} ({kappa_label(pooled_kappa)})")

    simple_avg_kappa = sum(per_construct_kappa.values()) / len(per_construct_kappa)
    log(f"  (For reference: simple average of the 4 per-construct kappas = {simple_avg_kappa:.3f})")

    # ---- bat_score agreement ----
    if "bat_score" in a_df.columns and "bat_score" in b_df.columns:
        log("\n" + "=" * 70)
        log("STEP 5: bat_score AGREEMENT")
        log("=" * 70)
        sa, sb = a_df["bat_score"], b_df["bat_score"]
        exact = (sa == sb)
        diff = (sa - sb).abs()
        log(f"  Exact match: {exact.sum()}/{len(sa)} ({exact.mean()*100:.1f}%)")
        log(f"  Mean absolute difference: {diff.mean():.2f}")

    # ---- Summary table ----
    log("\n" + "=" * 70)
    log("SUMMARY")
    log("=" * 70)
    for construct in CONSTRUCTS:
        log(f"  {construct:5s}: kappa = {per_construct_kappa[construct]:.3f} "
            f"({kappa_label(per_construct_kappa[construct])})")
    log(f"  {'POOLED':5s}: kappa = {pooled_kappa:.3f} ({kappa_label(pooled_kappa)}), "
        f"agreement = {pooled_pct:.1f}%")

    # ---- Write outputs ----
    with open(OUTPUT_REPORT, "w") as f:
        f.write("\n".join(lines))

    if disagreement_rows:
        pd.DataFrame(disagreement_rows).to_csv(OUTPUT_DISAGREEMENTS_CSV, index=False)
        log(f"\nDisagreement detail written to: {OUTPUT_DISAGREEMENTS_CSV} "
            f"({len(disagreement_rows)} disagreement rows)")
    else:
        log("\nNo disagreements found — no disagreement CSV written.")

    log(f"Full report written to: {OUTPUT_REPORT}")


if __name__ == "__main__":
    main()

TWO-WAY AGREEMENT ANALYSIS: Nadia vs. Jihyun

--- Loading files & normalizing values ---
Loading Nadia (merged_post40_Nadia.csv)...
Loading Jihyun (merged_post40_jiHyun.csv)...

STEP 1: SANITY CHECK — do both files cover the same posts?
  Nadia vs Jihyun: MATCH
  Row count [Nadia]: 40
  Row count [Jihyun]: 40

  Using 40 common posts for the analysis below.

STEP 2: PER-CONSTRUCT YES/NO COUNTS & AGREEMENT

--- EX ---
  Nadia   : YES= 11  NO= 29
  Jihyun  : YES= 14  NO= 26
  Agreement: 33/40 (82.5%)   Cohen's kappa = 0.595 (moderate)

--- EMO ---
  Nadia   : YES= 13  NO= 27
  Jihyun  : YES= 13  NO= 27
  Agreement: 34/40 (85.0%)   Cohen's kappa = 0.658 (substantial)

--- COG ---
  Nadia   : YES=  7  NO= 33
  Jihyun  : YES=  6  NO= 34
  Agreement: 35/40 (87.5%)   Cohen's kappa = 0.541 (moderate)

--- MD ---
  Nadia   : YES=  4  NO= 36
  Jihyun  : YES= 11  NO= 29
  Agreement: 33/40 (82.5%)   Cohen's kappa = 0.453 (moderate)

STEP 3: PER-POST EXACT MATCH (all 4 constructs agree, all-or-noth

Merge old labeled files and new labeled files 40 samples for each annotator, to do the overall inter-rater agreement

have to create merge csv files
bat_post20_LLM.csv + labeling_check_llm.xlsx - merged_post40_llm.csv
bat_post20_Nadia.csv + labeling_check_Nadia.xlsx - merged_post_Nadia.csv
bat_post20_jihyun.csv + labeling_check_jiHyun.xlsx- merged_post_jiHyun.csv



In [2]:
"""
Merge Old (20-post) + New (20-post) BAT Annotation Files -> 40-post files
==========================================================================
For each annotator, combines their original pilot file (20 posts) with
their new "labeling_check" file (20 posts) into a single 40-row CSV,
ready for the overall inter-rater agreement analysis
(bat_agreement_analysis.py).

Mappings:
  bat_post20_LLM.csv          + labeling_check_llm.xlsx    -> merged_post40_llm.csv
  bat_post20_Nadia_.csv       + labeling_check_Nadia.xlsx  -> merged_post40_Nadia.csv
  bat_post20_jihyun_.csv      + labeling_check_jiHyun.xlsx -> merged_post40_jiHyun.csv

Checks performed before merging:
  1. Both files load and have a comparable column set.
  2. No overlapping post_id/comment_id between old and new (would mean
     the same post got annotated twice under this annotator) - flagged,
     not silently deduped, since a genuine duplicate is a data problem
     to look at rather than the script's call to resolve.
  3. Row count sanity: reports old count, new count, merged count.
  4. Normalizes 0/1 -> NO/YES in EX/EMO/COG/MD (same normalization used
     in the agreement script), so downstream analysis doesn't choke on
     mixed encodings.

Usage:
    python3 merge_bat_annotations.py
Edit the MERGE_JOBS list below if filenames/paths change.
"""

import pandas as pd

CONSTRUCTS = ["EX", "EMO", "COG", "MD"]

# Each job: (annotator label, old csv path, new xlsx path, output csv path)
MERGE_JOBS = [
    ("LLM",    "bat_post20_LLM.csv",     "labeling_check_llm.xlsx",    "merged_post40_llm.csv"),
    ("Nadia",  "bat_post20_Nadia.csv",  "labeling_check_Nadia.xlsx",  "merged_post40_Nadia.csv"),
    ("Jihyun", "bat_post20_jihyun.csv", "labeling_check_jiHyun.xlsx", "merged_post40_jiHyun.csv"),
]


def load_any(path):
    """Load a CSV or XLSX into a DataFrame."""
    if path.lower().endswith(".xlsx") or path.lower().endswith(".xls"):
        return pd.read_excel(path)
    return pd.read_csv(path)


def normalize_constructs(df, label):
    """Normalize EX/EMO/COG/MD values to YES/NO, warn on anything unexpected."""
    normalize_map = {"YES": "YES", "NO": "NO", "1": "YES", "0": "NO",
                      "1.0": "YES", "0.0": "NO"}
    for c in CONSTRUCTS:
        if c not in df.columns:
            print(f"  WARNING [{label}]: column '{c}' missing from this file.")
            continue
        raw = df[c].astype(str).str.strip().str.upper()
        zero_one = raw.isin(["0", "1", "0.0", "1.0"]).sum()
        if zero_one:
            print(f"  NOTE [{label}]: normalized {zero_one} cell(s) coded as 0/1 in '{c}'.")
        mapped = raw.map(normalize_map)
        unmapped = mapped.isna() & df[c].notna()
        if unmapped.any():
            bad_vals = raw[unmapped].unique().tolist()
            print(f"  WARNING [{label}]: unrecognized values in '{c}': {bad_vals} "
                  f"(left as-is, not normalized).")
            mapped[unmapped] = df[c][unmapped]
        df[c] = mapped
    return df


def make_key(df):
    df = df.copy()
    df["comment_id"] = df["comment_id"].fillna("NONE") if "comment_id" in df.columns else "NONE"
    df["key"] = df["post_id"].astype(str) + "_" + df["comment_id"].astype(str)
    return df


def merge_one(label, old_path, new_path, out_path):
    print(f"\n=== {label}: {old_path} + {new_path} -> {out_path} ===")

    old_df = load_any(old_path)
    new_df = load_any(new_path)

    old_df = make_key(old_df)
    new_df = make_key(new_df)

    print(f"  Old file: {len(old_df)} rows, columns: {old_df.columns.tolist()}")
    print(f"  New file: {len(new_df)} rows, columns: {new_df.columns.tolist()}")

    # Column consistency check (core columns only, ignore ordering)
    core_cols = {"post_id", "comment_id", "text", "EX", "EMO", "COG", "MD", "bat_score"}
    missing_old = core_cols - set(old_df.columns)
    missing_new = core_cols - set(new_df.columns)
    if missing_old:
        print(f"  WARNING: old file missing expected columns: {missing_old}")
    if missing_new:
        print(f"  WARNING: new file missing expected columns: {missing_new}")

    # Overlap check: same post shouldn't appear in both old and new
    overlap = set(old_df["key"]) & set(new_df["key"])
    if overlap:
        print(f"  WARNING: {len(overlap)} post(s) appear in BOTH old and new "
              f"files for {label}: {sorted(overlap)}")
        print("  These will appear twice in the merged file unless you "
              "resolve them manually before/after merging.")
    else:
        print("  No overlap between old and new posts. Good.")

    # Normalize construct values in both
    old_df = normalize_constructs(old_df, f"{label}-old")
    new_df = normalize_constructs(new_df, f"{label}-new")

    # Tag provenance (useful for later filtering/debugging), then merge
    old_df["batch"] = "old_20"
    new_df["batch"] = "new_20"

    # Align columns: keep union of columns from both, in old_df's order first
    all_cols = list(old_df.columns) + [c for c in new_df.columns if c not in old_df.columns]
    old_aligned = old_df.reindex(columns=all_cols)
    new_aligned = new_df.reindex(columns=all_cols)

    merged = pd.concat([old_aligned, new_aligned], ignore_index=True)

    # Duplicate key check on the merged result
    dup_keys = merged[merged.duplicated(subset="key", keep=False)]
    if not dup_keys.empty:
        print(f"  WARNING: {dup_keys['key'].nunique()} duplicate post key(s) "
              f"in the merged file (see 'key' column).")

    print(f"  Merged: {len(merged)} rows "
          f"({len(old_df)} old + {len(new_df)} new).")

    # Drop the helper 'key' column from the saved file (keep post_id/comment_id instead)
    merged_out = merged.drop(columns=["key"])
    merged_out.to_csv(out_path, index=False)
    print(f"  Saved -> {out_path}")

    return merged_out


def main():
    results = {}
    for label, old_path, new_path, out_path in MERGE_JOBS:
        results[label] = merge_one(label, old_path, new_path, out_path)

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    for label, df in results.items():
        print(f"  {label:8s}: {len(df)} total rows -> {label}")


if __name__ == "__main__":
    main()


=== LLM: bat_post20_LLM.csv + labeling_check_llm.xlsx -> merged_post40_llm.csv ===
  Old file: 20 rows, columns: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'key']
  New file: 20 rows, columns: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'key']
  No overlap between old and new posts. Good.
  Merged: 40 rows (20 old + 20 new).
  Saved -> merged_post40_llm.csv

=== Nadia: bat_post20_Nadia.csv + labeling_check_Nadia.xlsx -> merged_post40_Nadia.csv ===
  Old file: 20 rows, columns: ['row_type', 'post_id', 'comment_id', 'text', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'key']
  New file: 20 rows, columns: ['row_type', 'post_id', 'comment_id', 'text', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning